In [0]:
import requests
import json

url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

dados = requests.get(url).json()

print(json.dumps(dados[0], indent=4, ensure_ascii=False))

In [0]:
import requests
import json

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

dados = requests.get(url).json()

rows = [
    Row(payload=json.dumps(registro, ensure_ascii=False))
    for registro in dados
]

df_bronze = (
    spark.createDataFrame(rows)
         .withColumn("date_ingestion", current_timestamp())
)

(
    df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gs_carbon.bronze.ibge_municipios")
)

In [0]:
%sql
SELECT * FROM gs_carbon.bronze.ibge_municipios